In [ ]:
import sqlite3
import pandas as pd

# Bağlantıyı kur
conn = sqlite3.connect("movies_analytics.db")
cur = conn.cursor()

# Eski tablolar varsa sıfırlayalım (Hata almamak için)
cur.execute("DROP TABLE IF EXISTS watch_history;")
cur.execute("DROP TABLE IF EXISTS movies;")

# Tabloları Oluşturma
cur.execute("""
CREATE TABLE movies (
    movie_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    genre TEXT NOT NULL,
    release_year INTEGER,
    imdb_rating REAL,
    budget_mil REAL
);
""")

cur.execute("""
CREATE TABLE watch_history (
    watch_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    movie_id INTEGER,
    watch_date TEXT,
    watch_time_minutes INTEGER,
    FOREIGN KEY (movie_id) REFERENCES movies (movie_id)
);
""")

# Güncellenmiş Film Listesi (Yeni Sci-Fi ve Action Filmleri Dahil)
movies_data = [
    ('Inception', 'Sci-Fi', 2010, 8.8, 160.0),
    ('Interstellar', 'Sci-Fi', 2014, 8.7, 165.0),
    ('The Dark Knight', 'Action', 2008, 9.0, 185.0),
    ('Pulp Fiction', 'Crime', 1994, 8.9, 8.0),
    ('The Matrix', 'Sci-Fi', 1999, 8.7, 63.0),
    ('Parasite', 'Drama', 2019, 8.5, 11.4),
    ('Whiplash', 'Drama', 2014, 8.5, 3.3),
    ('Avengers: Endgame', 'Action', 2019, 8.4, 356.0),
    # --- YENİ EKLENEN FİLMLER ---
    ('Dune: Part Two', 'Sci-Fi', 2024, 8.6, 190.0),
    ('Blade Runner 2049', 'Sci-Fi', 2017, 8.0, 150.0),
    ('Arrival', 'Sci-Fi', 2016, 7.9, 47.0)
]

cur.executemany("INSERT INTO movies (title, genre, release_year, imdb_rating, budget_mil) VALUES (?, ?, ?, ?, ?)", movies_data)

# İzleme geçmişi verileri
history_data = [
    (101, 1, '2026-01-10', 148), # Inception
    (101, 2, '2026-01-15', 169), # Interstellar
    (102, 3, '2026-02-01', 152), # The Dark Knight
    (103, 1, '2026-02-10', 148),
    (103, 5, '2026-02-12', 136), # The Matrix
    (104, 6, '2026-02-20', 132), # Parasite
    (101, 5, '2026-03-01', 136)  # The Matrix
]

cur.executemany("INSERT INTO watch_history (user_id, movie_id, watch_date, watch_time_minutes) VALUES (?, ?, ?, ?)", history_data)

conn.commit()
print("[+] Veritabanı ve örnek veriler başarıyla güncellendi!")

In [ ]:
# 1. En Yüksek İMDB Puanına Sahip Türlerin Ortalama Bütçe Analizi
query1 = """
SELECT
    genre,
    ROUND(AVG(imdb_rating), 2) as avg_rating,
    ROUND(AVG(budget_mil), 2) as avg_budget_million
FROM movies
GROUP BY genre
ORDER BY avg_rating DESC;
"""

print("=== TÜR BAZLI PUAN VE BÜTÇE ANALİZİ ===")
df1 = pd.read_sql_query(query1, conn)
display(df1)

# 2. İzleme Geçmişi ve Film Detaylarının Birleştirilmesi (JOIN)
query2 = """
SELECT
    wh.user_id,
    m.title,
    m.genre,
    wh.watch_date
FROM watch_history wh
JOIN movies m ON wh.movie_id = m.movie_id
ORDER BY wh.user_id;
"""

print("\n=== KULLANICI İZLEME GEÇMİŞİ (JOIN) ===")
df2 = pd.read_sql_query(query2, conn)
display(df2)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("movies_analytics.db")

def recommend_movie_smart(user_id, top_n=5):
    # 1. Kullanıcının daha önce izlediği türleri bul
    query_genres = f"""
    SELECT DISTINCT m.genre
    FROM watch_history wh
    JOIN movies m ON wh.movie_id = m.movie_id
    WHERE wh.user_id = {user_id};
    """
    watched_genres = pd.read_sql_query(query_genres, conn)['genre'].tolist()

    if not watched_genres:
        print(f"Kullanıcı {user_id} için henüz izleme geçmişi yok.")
        return

    # 2. Kullanıcının izlediği türlerdeki, izlemediği YÜKSEK PUANLI filmleri çek
    genres_placeholder = "','".join(watched_genres)

    # RANDOM() ekleyerek her çalıştırmada en iyi top_n film arasından rastgele getiriyoruz!
    rec_query = f"""
    WITH TopMovies AS (
        SELECT title, genre, release_year, imdb_rating
        FROM movies
        WHERE genre IN ('{genres_placeholder}')
          AND movie_id NOT IN (SELECT movie_id FROM watch_history WHERE user_id = {user_id})
        ORDER BY imdb_rating DESC
        LIMIT {top_n}  -- En yüksek IMDb puanlı ilk 5 filmi al
    )
    SELECT * FROM TopMovies
    ORDER BY RANDOM() -- En iyi 5 film arasından RASTGELE birini seç
    LIMIT 1;
    """

    recommendation = pd.read_sql_query(rec_query, conn)

    print(f"🎬 === KULLANICI {user_id} İÇİN ÖNERİ ===")
    print(f"📌 İzlediği Favori Türler: {watched_genres}")

    if not recommendation.empty:
        rec = recommendation.iloc[0]
        print(f"✨ Önerilen Film: {rec['title']} ({rec['release_year']})")
        print(f"🎭 Tür: {rec['genre']} | ⭐ IMDb Puanı: {rec['imdb_rating']}\n")
    else:
        print("Öneri için uygun yeni film bulunamadı.")

# Kodu birkaç kez çalıştırarak her seferinde farklı öneri geldiğini görebilirsin:
recommend_movie_smart(101)

In [ ]:
import sqlite3
import requests

conn = sqlite3.connect("movies_analytics.db")
cur = conn.cursor()

# 150+ Filmlik Genişletilmiş Canlı Film Listesi
movie_list = [
    # SCI-FI / BİLİM KURGU
    "Inception", "Interstellar", "The Matrix", "Dune", "Avatar", "Blade Runner 2049",
    "Arrival", "Tenet", "The Martian", "Gravity", "Ex Machina", "Alien", "Terminator 2",
    "Star Wars: Episode V", "Jurassic Park", "Ready Player One", "Edge of Tomorrow",
    "Minority Report", "12 Monkeys", "District 9", "Dune: Part Two", "2001: A Space Odyssey",

    # ACTION & ADVENTURE / AKSİYON & MACERA
    "The Dark Knight", "Gladiator", "Mad Max: Fury Road", "Avengers: Endgame",
    "Top Gun: Maverick", "John Wick", "Die Hard", "Casino Royale", "Batman Begins",
    "Spider-Man: Across the Spider-Verse", "Logan", "Raiders of the Lost Ark",
    "Mission: Impossible - Fallout", "V for Vendetta", "Kill Bill: Vol. 1", "300",
    "Gladiator II", "Dunkirk", "The Revenant", "Ford v Ferrari",

    # DRAMA
    "The Shawshank Redemption", "Forrest Gump", "Fight Club", "The Godfather",
    "Pulp Fiction", "Whiplash", "Oppenheimer", "Parasite", "Good Will Hunting",
    "The Green Mile", "12 Angry Men", "Schindler's List", "The Prestige", "Memento",
    "A Beautiful Mind", "The Social Network", "La La Land", "The Truman Show",
    "Dead Poets Society", "The Pianist", "American History X", "Whiplash",

    # CRIME & THRILLER / SUÇ & GERİLİM
    "Se7en", "The Silence of the Lambs", "The Departed", "Joker", "Goodfellas",
    "No Country for Old Men", "Heat", "Catch Me If You Can", "Shutter Island",
    "Prisoners", "Nightcrawler", "The Usual Suspects", "Zodiac", "Gone Girl",
    "Sicario", "Taxi Driver", "Léon: The Professional", "Fargo", "Scarface",

    # ANIMATION & FANTASY / ANİMASYON & FANTEZİ
    "Spirited Away", "WALL-E", "Coco", "Toy Story", "Up", "Ratatouille", "Inside Out",
    "Finding Nemo", "Shrek", "How to Train Your Dragon", "Spider-Man: Into the Spider-Verse",
    "The Lion King", "Princess Mononoke", "Your Name", "The Lord of the Rings: The Fellowship of the Ring",
    "The Lord of the Rings: The Two Towers", "The Lord of the Rings: The Return of the King",

    # COMEDY & ROMANCE / KOMEDİ & ROMANTİK
    "The Grand Budapest Hotel", "Superbad", "The Hangover", "Back to the Future",
    "Eternal Sunshine of the Spotless Mind", "500 Days of Summer", "Amélie",
    "Groundhog Day", "Knives Out", "Crazy, Stupid, Love", "The Wolf of Wall Street",

    # HORROR & MYSTERY / KORKU & GİZEM
    "The Shining", "Get Out", "A Quiet Place", "Hereditary", "Psycho", "The Sixth Sense",
    "Alien", "The Thing", "The Conjuring", "Black Swan", "Shutter Island"
]

# Çakışan mükerrer isimleri listeden temizleyelim
movie_list = list(set(movie_list))

print(f"🔄 Toplam {len(movie_list)} benzersiz film için canlı veri çekiliyor...\n")

added_count = 0

for idx, movie_name in enumerate(movie_list, 1):
    url = f"http://www.omdbapi.com/?t={movie_name}&apikey=trilogy"

    try:
        response = requests.get(url, timeout=5)
        data = response.json()

        if data.get("Response") == "True":
            title = data.get("Title")
            genre = data.get("Genre").split(",")[0].strip()  # İlk ana türü alır
            release_year = int(data.get("Year")[:4])

            imdb_str = data.get("imdbRating")
            imdb_rating = float(imdb_str) if imdb_str and imdb_str != "N/A" else 0.0
            budget_mil = 100.0  # Varsayılan bütçe

            cur.execute("SELECT * FROM movies WHERE title = ?", (title,))
            if not cur.fetchone():
                cur.execute("""
                INSERT INTO movies (title, genre, release_year, imdb_rating, budget_mil)
                VALUES (?, ?, ?, ?, ?)
                """, (title, genre, release_year, imdb_rating, budget_mil))
                added_count += 1
                print(f"[{idx}/{len(movie_list)}] ✅ Eklendi: {title} | Tür: {genre} | IMDb: {imdb_rating}")
            else:
                print(f"[{idx}/{len(movie_list)}] ℹ️ Zaten var: {title}")

    except Exception as e:
        print(f"[{idx}/{len(movie_list)}] ❌ Hata ({movie_name}): {e}")

conn.commit()
print(f"\n[+] İşlem tamamlandı! Toplam {added_count} yeni canlı film eklendi.")